# Kaggle Dataset Explorer
## Schnelle Datenanalyse für Portfolio-Projekte

Dieses Notebook hilft dir dabei, Kaggle-Datasets schnell zu explorieren und zu entscheiden, ob sie für dein Portfolio-Projekt geeignet sind.

---

## 1. Setup & Daten laden



In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from core.data import load_from_kaggle

# Styling für bessere Visualisierungen
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# GeoPy with Nominatim to estimate lat, lon from city name
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import os
import time
import requests     # um die OpenAPI für Höhenangaben bei lat, lon abzufragen

# Meteostat for weather data
from datetime import date
import meteostat as ms

# Models
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.inspection import permutation_importance

# import module for Score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, roc_curve, auc
from sklearn.model_selection import cross_val_score

In [ ]:
# # ===== HIER DATASET-LINK EINFÜGEN =====
# full_link = r"https://www.kaggle.com/datasets/orvile/european-cities-weather-prediction-dataset"
# dataset_link = full_link.split("/datasets/")[-1]
# # ======================================

# destination = "../data/raw"
# dataset_name = dataset_link.split("/")[-1]

# print(f"📦 Lade Dataset: {dataset_name}")
# files = load_from_kaggle(
#     dataset_link=dataset_link, 
#     destination=destination,
# )

# print(f"✅ {len(files)} Datei(en) gefunden:")
# for i, file in enumerate(files, 1):
#     print(f"   {i}. {file}")




## 2. Daten einlesen & erste Inspektion



In [ ]:

# Erste CSV-Datei laden (oder bei Bedarf anpassen)
# file_path = "/".join([destination, dataset_name, files[0]])
# df = pd.read_csv(file_path)

# print(f"📊 Geladene Datei: {files[0]}")
# print(f"📏 Shape: {df.shape[0]:,} Zeilen × {df.shape[1]} Spalten\n")

file_weather_light = '../data/raw/european-cities-weather-prediction-dataset/weather_prediction_dataset_light.csv'
file_picnic_label  = '../data/raw/european-cities-weather-prediction-dataset/weather_prediction_picnic_labels.csv'
file_weather_full  = '../data/raw/european-cities-weather-prediction-dataset/weather_prediction_dataset.csv'
# file_weather_FS    = '../data/raw/european-cities-weather-prediction-dataset/weather_prediction_dataset_FS.csv'
file_weather_FS    = '../data/raw/european-cities-weather-prediction-dataset/weather_prediction_dataset_FS_noROMA.csv'

# Load the datasets from the specified CSV files
weather_light = pd.read_csv(file_weather_light, 
                            delimiter=',', encoding='ascii')

print(f"📊 Geladene Datei: {file_weather_light}")
print(f"📏 Shape: {weather_light.shape[0]:,} Zeilen × {weather_light.shape[1]} Spalten\n")

picnic_labels = pd.read_csv(file_picnic_label, 
                             delimiter=',', encoding='ascii')
print(f"📊 Geladene Datei: {file_picnic_label}")
print(f"📏 Shape: {picnic_labels.shape[0]:,} Zeilen × {picnic_labels.shape[1]} Spalten\n")

# For extended analysis, you could also load the complete dataset if needed
weather_full = pd.read_csv(file_weather_full, 
                           delimiter=',', encoding='ascii')
print(f"📊 Geladene Datei: {file_weather_full}")
print(f"📏 Shape: {weather_full.shape[0]:,} Zeilen × {weather_full.shape[1]} Spalten\n")

# For extended analysis, you could also load the complete dataset if needed
weather_FS = pd.read_csv(file_weather_FS, 
                           delimiter=',', encoding='ascii')
print(f"📊 Geladene Datei: {file_weather_FS}")
print(f"📏 Shape: {weather_FS.shape[0]:,} Zeilen × {weather_FS.shape[1]} Spalten\n")

# Convert the DATE column to datetime. The DATE is given as an integer (likely in YYYYMMDD format)
for df in [weather_light, picnic_labels, weather_full, weather_FS]:
    df['DATE'] = pd.to_datetime(df['DATE'].astype(str), format='%Y%m%d', errors='coerce')

print('Data loaded and DATE column converted to datetime.')

weather_FS.sample(3)




---

## 3. Datenqualität & Struktur



In [ ]:

print("=" * 80)
print("📋 DATENÜBERSICHT weather_light")
print("=" * 80)

# Grundlegende Statistiken
print(f"\n🔢 Dimensionen: {weather_light.shape[0]:,} Zeilen × {weather_light.shape[1]} Spalten")
print(f"🔄 Duplikate: {weather_light.duplicated().sum():,} ({weather_light.duplicated().sum()/len(weather_light)*100:.2f}%)")
print(f"💾 Speichernutzung: {weather_light.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("=" * 80)
print("📋 DATENÜBERSICHT picnic_labels")
print("=" * 80)

# Grundlegende Statistiken
print(f"\n🔢 Dimensionen: {picnic_labels.shape[0]:,} Zeilen × {picnic_labels.shape[1]} Spalten")
print(f"🔄 Duplikate: {weather_light.duplicated().sum():,} ({picnic_labels.duplicated().sum()/len(picnic_labels)*100:.2f}%)")
print(f"💾 Speichernutzung: {picnic_labels.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("=" * 80)
print("📋 DATENÜBERSICHT weather_full")
print("=" * 80)

# Grundlegende Statistiken
print(f"\n🔢 Dimensionen: {weather_full.shape[0]:,} Zeilen × {weather_full.shape[1]} Spalten")
print(f"🔄 Duplikate: {weather_full.duplicated().sum():,} ({weather_full.duplicated().sum()/len(weather_full)*100:.2f}%)")
print(f"💾 Speichernutzung: {weather_full.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("=" * 80)
print("📋 DATENÜBERSICHT weather_FS")
print("=" * 80)

# Grundlegende Statistiken
print(f"\n🔢 Dimensionen: {weather_FS.shape[0]:,} Zeilen × {weather_FS.shape[1]} Spalten")
print(f"🔄 Duplikate: {weather_FS.duplicated().sum():,} ({weather_FS.duplicated().sum()/len(weather_FS)*100:.2f}%)")
print(f"💾 Speichernutzung: {weather_FS.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


In [ ]:
# Func Def: Overview(df) Übersicht entwerfen, um immer mal wieder einen schnellen Überblick über die wichtigsten "Metadaten" zu bekommen
def overview(df):
    '''
    Erstelle einen Überblick über einige wichtige Eigenschaften der Spalten eines DataFrames.
    VARs
        df: Der zu betrachtende DataFrame
    RETURNS:
        None
    '''
    df = df.copy()
    display(pd.DataFrame({'dtype': df.dtypes,
                          'total': df.count(),
                          'missing_n': df.isna().sum(),
                          'missing_%': df.isna().mean()*100,
                          'uniques_n': df.nunique(),
                          'uniques': [df[col].unique() for col in df.columns]
                         }))

In [ ]:
overview(weather_FS)

In [ ]:
# overview(weather_light)
# overview(picnic_labels)
# overview(weather_full)
# for name in weather_light.columns:
#     print(name)
# for name in weather_full.columns:
#     print("'" + name + "',")
for name in picnic_labels.columns:
    print("'" + name + "',")


In [ ]:
# MultiIndex-Spalten in flache Strings umwandeln (z. B. "Basel_A_bis_2023")
from enum import unique


c_names = picnic_labels.columns
# Spaltennamen aufteilen: nur am ersten '_'
split_cols = [st.split('_', maxsplit=1) for st in c_names[2:]]
c_names[2]
split_cols[0][0]
len(split_cols)
cit = [split_cols[i][0] for i in range(len(split_cols))]
set(cit)
# para = [split_cols[i][1] for i in range(len(split_cols))]
# set(para)
# pd.DataFrame(cit).unique()
# weather_full.columns = ['_'.join(col).strip() for col in weather_full.columns.values]

# # Spaltennamen aufteilen: nur am ersten '_'
# split_cols = weather_full.columns.str.split('_', n=1, expand=True)
# type(split_cols)
# split_cols
# city = split_cols[0].unique()
# parameter = split_cols[1].unique()
# city

In [ ]:
col_names = ['Date', 'Month',
             'cloud_cover', 'humidity', 'pressure', 'global_radiation',
             'precipitation', 'sunshine', 'temp_mean', 'temp_min', 'temp_max',
             'lat', 'lon', 'city']

cities = ['BASEL', 'BUDAPEST', 'DE', 'DRESDEN', 'DUSSELDORF', 'HEATHROW', 'KASSEL', 'LJUBLJANA',
          'MAASTRICHT', 'MALMO', 'MONTELIMAR', 'MUENCHEN', 'OSLO', 'PERPIGNAN',
          'SONNBLICK', 'STOCKHOLM', 'TOURS']

df_basel = weather_full.loc[:,['DATE', 'MONTH', 'BASEL_cloud_cover', 'BASEL_humidity', 'BASEL_pressure', 'BASEL_global_radiation',
                               'BASEL_precipitation', 'BASEL_sunshine', 'BASEL_temp_mean', 'BASEL_temp_min', 'BASEL_temp_max']]
df_basel['lat'] = 47.5596
df_basel['lon'] = 7.5886
df_basel['city'] = 'Basel'
df_basel.columns = col_names
df_basel

In [ ]:
weather_FS

In [ ]:
weather_FS.city.unique()

In [ ]:

def get_coordinates(city_name, user_agent="MeineApp/1.0 (meine@email.de)"):
    base_url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": city_name,
        "format": "json",
        "limit": 1,
    }
    headers = {
        "User-Agent": user_agent,
    }

    try:
        response = requests.get(base_url, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()
        if data:
            return float(data[0]["lat"]), float(data[0]["lon"])
        else:
            return None, None
    except Exception as e:
        print(f"Fehler: {e}")
        return None, None


In [ ]:
# Beispielaufruf
lat, lon = get_coordinates("Berlin")
print(f"Lat: {lat}, Lon: {lon}")

In [ ]:
unique_cities = weather_FS['city'].unique()

geolocator = Nominatim(user_agent="geoapi")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def get_coordinates(city):
    try:
        location = geocode(city)
        if location:
            return (location.latitude, location.longitude)
        else:
            return (None, None)
    except:
        return (None, None)

city_coords = {}
for city in unique_cities:
    city_coords[city] = get_coordinates(city)
    lat, lon = city_coords[city]
    print(f"City: {city}, Lat: {lat}, Lon: {lon}")

In [ ]:
import requests
from requests.exceptions import Timeout

def get_elevation(lat, lon, timeout=10):
    url = f"https://api.open-elevation.com/api/v1/lookup?locations={lat},{lon}"
    try:
        response = requests.get(url, timeout=timeout)
        if response.status_code == 200:
            data = response.json()
            return data["results"][0]["elevation"]
        else:
            print(f"Fehler: HTTP-Statuscode {response.status_code}")
            return None
    except Timeout:
        print(f"Timeout: Die Anfrage hat länger als {timeout} Sekunden gedauert.")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Fehler bei der Anfrage: {e}")
        return None

# Beispiel: Höhe für Berlin abfragen
elevation = get_elevation(52.5200, 13.4050, timeout=10)
if elevation is not None:
    print(f"Höhe: {elevation} Meter")

In [ ]:

# def get_elevation(lat, lon):
#     url = f"https://api.open-elevation.com/api/v1/lookup?locations={lat},{lon}"
#     response = requests.get(url).json()
#     return response["results"][0]["elevation"]

# Beispiel
# lat, lon = 52.5200, 13.4050  # Berlin
# elevation = get_elevation(lat, lon)
# print(f"Höhe: {elevation} Meter")

In [ ]:

# Da alle Zeilen pro Stadt die gleiche Elevation haben, hole Elevation pro Stadt
city_elevations = {}
for city in weather_FS['city'].unique():
    print('City:', city)
    lat, lon = city_coords[city]
    try:
        elevation = get_elevation(lat, lon)
        city_elevations[city] = elevation
        print(f"Elevation für {city}: {elevation} m")
    except Exception as e:
        print(f"Fehler bei {city}: {e}")
        city_elevations[city] = None


In [ ]:
# Füge lat, lon zu weather_FS hinzu
weather_FS['lat'] = weather_FS['city'].map(lambda x: city_coords[x][0])
weather_FS['lon'] = weather_FS['city'].map(lambda x: city_coords[x][1])
# Füge Elevation zu weather_FS hinzu
# weather_FS['elevation'] = weather_FS['city'].map(city_elevations)

weather_FS.head(3)

In [ ]:
# city_elevations = {}
# city_elevations['BASEL'] = 263
# city_elevations['BUDAPEST'] = 128
# city_elevations['DE_BILT'] = 3
# city_elevations['DRESDEN'] = 122
# city_elevations['DUSSELDORF'] = 44
# city_elevations['HEATHROW'] = 16
# city_elevations['KASSEL'] = 170
# city_elevations['LJUBLJANA'] = 298
# city_elevations['MAASTRICHT'] = 56
# city_elevations['MALMO'] = 14
# city_elevations['MONTELIMAR'] = 90
# city_elevations['MUENCHEN'] = 523
# city_elevations['OSLO'] = 19
# city_elevations['PERPIGNAN'] = 40
# city_elevations['SONNBLICK'] = 2427
# city_elevations['STOCKHOLM'] = 28
# city_elevations['TOURS'] = 55


In [ ]:
# Füge Elevation zu weather_FS hinzu
weather_FS['elevation'] = weather_FS['city'].map(city_elevations)

weather_FS.head(3)

In [ ]:
# Speichere den DataFrame als CSV-Datei
# weather_FS.to_csv('../data/processed/weather_FS_lat_lon.csv', index=False)
# weather_FS.to_csv('../data/processed/weather_FS_lat_lon_elevation.csv', index=False)
# print("DataFrame gespeichert als: data/processed/weather_FS_lat_lon_elevation.csv")

### Load Data - Weather_FS and Meteostat_all

In [ ]:
# Load Data from file
file_weather_FS    = '../data/processed/weather_FS_lat_lon_elevation.csv'
file_weather_meteostat    = '../data/processed/weather_meteostat_lat_lon_elevation.csv'

# For extended analysis, you could also load the complete dataset if needed
weather_FS = pd.read_csv(file_weather_FS, 
                           delimiter=',', encoding='ascii', parse_dates=['DATE'])

# For extended analysis, you could also load the complete dataset if needed
meteostat_all = pd.read_csv(file_weather_meteostat, 
                           delimiter=',', encoding='ascii', parse_dates=['DATE'])

In [ ]:
# Funktion zum Plotten der nach der Zielvariable gruppierten Verteilungen für numerischen Features
def numplots(col, data=weather_FS, key='picnic_weather', save_plot=False):
    '''Plot a boxplot, histogram and kernel density estimation (kde) plot, grouped by 'key'
    ARGS
        col: Column to plot
        data: DataFrame (default: eda)
        key: Column to group by (default: 'IsBadBuy')
        save_plot: default: False. If True save plot as png, filename = f"numplot_{col}.png"
    RETURN: None
    '''
    fig, ax = plt.subplots(ncols=3, figsize=(16,3))
    sns.boxplot(data=data, y=col, x=key, ax=ax[0])
    data.groupby(key)[col].plot(kind='hist', bins=20, ax=ax[1], alpha=0.5, density=True) # Der Parameter density=True ist v.a. bei unausgeglichenen Zielvariablen nützlich
    data.groupby(key)[col].plot(kind='kde', ax=ax[2])

    if save_plot:
        fig.savefig(f"numplot_{col}.png")

In [ ]:
# numplots('precipitation')
# numplots('temp_mean')
# numplots('temp_max')
# numplots('lon')
numplots('elevation')


In [ ]:
# from turtle import title


def numplot_city(city, data=weather_FS, key='picnic_weather', var_list=['lat', 'lon', 'elevation'], save_plot=False):
    '''Plot a boxplot, histogram and kernel density estimation (kde) plot, grouped by 'key'
    ARGS
        city: City of which data to plot
        data: DataFrame (default: eda)
        key: Column to group by (default: 'picnic_weather')
        var_list: List of columns to include in the plot
        save_plot: default: False. If True save plot as png, filename = f"numplot_{city}.png"
    RETURN: None
    '''
    df = data[data.city == city]
    fig, ax = plt.subplots(ncols=2, nrows=len(var_list), figsize=(16,3*len(var_list)))
    # title = f"Verteilungen der numerischen Features für {city}"
    # fig.suptitle(title, fontsize=16)
    for i, var in enumerate(var_list):
        if df[var].empty:
            print(f"Warnung: Keine Daten für {var} in {city}. Plot wird übersprungen.")
            continue
        sns.boxplot(data=df, y=var, x=key, ax=ax[i, 0])
        ax[i, 0].set_xlabel(f"")
        df.groupby(key)[var].plot(kind='hist', bins=20, ax=ax[i, 1], alpha=0.5, density=True) # Der Parameter density=True ist v.a. bei unausgeglichenen Zielvariablen nützlich
        ax[i, 1].set_xlabel(f"{var}")
        # df.groupby(key)[var].plot(kind='kde', ax=ax[i, 2])
    fig.tight_layout

    if save_plot:
        fig.savefig(f"..\\plots\\numplot_{city}.png")
        plt.close(fig)

In [ ]:
for cit in weather_FS.city.unique():
    print('City:', cit)
    numplot_city(cit, var_list=['temp_mean', 'temp_max', 
                                'humidity', 'sunshine', 'precipitation', 
                                ], save_plot=True)

# numplot_city('BASEL', var_list=['temp_mean', 'temp_max', 'temp_min', 
#                                 'humidity', 'sunshine', 'precipitation', 
#                                 'cloud_cover'], save_plot=True)

In [ ]:
# df1 = weather_FS.groupby('picnic_weather')['temp_max']
df1 = weather_FS.groupby('picnic_weather')['precipitation']
df1.plot(kind='hist', bins=20, alpha=0.5, density=True)
# df1.plot(kind='kde')

In [ ]:
df_filtered = weather_FS[weather_FS.picnic_weather == 1]
df_filtered#['picnic_weather'].value_counts()


In [ ]:
overview(weather_FS)

In [ ]:
def weather_meteostat_city(city_name):
    # Get coordinates for the city
    lat, lon = city_coords[city_name]
    
    # Define time period (e.g., last 5 years)
    end = date.today()
    start = date(end.year - 5, end.month, end.day)
    
    # Get weather data using Meteostat
    location = ms.Point(lat, lon)
    data = ms.daily(location, start, end)
    data = data.fetch()
    
    # Plot line chart including average, minimum and maximum temperature
    # data.plot(y=[ms.Parameter.TEMP, ms.Parameter.TMIN, ms.Parameter.TMAX])
    # plt.show()
    
    return data

In [ ]:
df_weather = weather_meteostat_city('BASEL')
print(df_weather)

In [ ]:
def weather_meteostat(lat, lon, save_csv=False, csv_path=None):
    # Specify location and time range
    POINT = ms.Point(lat, lon, 113)  # Try with your location
    START = date(2000, 1, 1)
    END = date(2010, 1, 1)

    # Get nearby weather stations
    stations = ms.stations.nearby(POINT, limit=4)

    # Get daily data & perform interpolation
    ts = ms.daily(stations, START, END)
    df = ms.interpolate(ts, POINT).fetch()

    # Save fetched data to CSV if requested
    if save_csv:
        if csv_path is None:
            csv_path = f'weather_meteostat_{lat:.4f}_{lon:.4f}.csv'
        df.to_csv(csv_path, index=True)
        print(f'Saved weather data to {csv_path}')

    # Plot line chart including average, minimum and maximum temperature
    df.plot(y=[ms.Parameter.TEMP, ms.Parameter.TMIN, ms.Parameter.TMAX])
    plt.show()

    return df

In [ ]:
def weather_meteostat2(city, save_csv=False, csv_path=None):
    # Use the city name to get coordinates and elevation
    if city not in city_coords:
        raise ValueError(f"Unknown city: {city}")

    lat, lon = city_coords[city]
    elevation = city_elevations.get(city, 113)

    # Specify location and time range
    POINT = ms.Point(lat, lon, elevation)
    START = date(2000, 1, 1)
    END = date(2010, 1, 1)

    # Get nearby weather stations
    stations = ms.stations.nearby(POINT, limit=4)

    # Get daily data & perform interpolation
    ts = ms.daily(stations, START, END)
    df = ms.interpolate(ts, POINT).fetch()

    # Reset index so the time index becomes a normal column and can be renamed
    df = df.reset_index()

    # Rename columns to match weather_FS naming
    df = df.rename(columns={
        'time': 'DATE',
        'temp': 'temp_mean',
        'tmin': 'temp_min',
        'tmax': 'temp_max',
        'rhum': 'humidity',
        'prcp': 'precipitation',
        'snwd': 'snowfall',
        'wspd': 'wind_speed',
        'pres': 'pressure',
        'tsun': 'sunshine',
        'cldc': 'cloud_cover'
    })

    # Convert sunshine from minutes to hours
    if 'sunshine' in df.columns:
        df['sunshine'] = df['sunshine'] / 60

    # Save fetched data to CSV if requested
    if save_csv:
        if csv_path is None:
            csv_path = f'weather_meteostat_{city.lower()}.csv'
        df.to_csv(csv_path, index=True)
        print(f'Saved weather data to {csv_path}')

    # Plot line chart including average, minimum and maximum temperature
    plot_columns = []
    if 'temp_mean' in df.columns:
        plot_columns.append('temp_mean')
    if 'temp_min' in df.columns:
        plot_columns.append('temp_min')
    if 'temp_max' in df.columns:
        plot_columns.append('temp_max')

    if plot_columns:
        df.plot(y=plot_columns)
        plt.show()

    return df

In [ ]:
import glob

def load_meteostat_city_csvs(csv_dir='../data/processed', file_pattern='weather_meteostat_*_2000-2010.csv'):
    """Load all Meteostat city CSV files and enrich with city, lat, lon, elevation."""
    file_paths = sorted(glob.glob(os.path.join(csv_dir, file_pattern)))
    if not file_paths:
        raise FileNotFoundError(f"No Meteostat CSV files found in {csv_dir} matching {file_pattern}")

    dfs = []
    for file_path in file_paths:
        df_city = pd.read_csv(file_path)

        # Normalize Date column
        if 'time' in df_city.columns:
            df_city = df_city.rename(columns={'time': 'DATE'})
        if 'DATE' not in df_city.columns:
            raise ValueError(f"DATE column not found in {file_path}")
        df_city['DATE'] = pd.to_datetime(df_city['DATE'], errors='coerce')

        # Remove unnamed index columns created by CSV export
        df_city = df_city.loc[:, [col for col in df_city.columns if not str(col).startswith('Unnamed')]]

        # Determine city name from file name when not already present
        if 'city' not in df_city.columns or df_city['city'].isna().all():
            basename = os.path.basename(file_path)
            city_name = basename.replace('weather_meteostat_', '').replace('_2000-2010.csv', '').upper()
            df_city['city'] = city_name
        else:
            city_name = str(df_city['city'].iloc[0]).upper()

        # Add latency, longitude and elevation from the existing lookup tables
        if 'lat' not in df_city.columns or 'lon' not in df_city.columns:
            coords = city_coords.get(city_name, (None, None))
            df_city['lat'] = coords[0]
            df_city['lon'] = coords[1]

        if 'elevation' not in df_city.columns:
            df_city['elevation'] = city_elevations.get(city_name)

        # Ensure lower-case standardized column names for the weather features
        df_city = df_city.rename(columns={
            'temp': 'temp_mean',
            'tmin': 'temp_min',
            'tmax': 'temp_max',
            'rhum': 'humidity',
            'prcp': 'precipitation',
            'snwd': 'snowfall',
            'wspd': 'wind_speed',
            'pres': 'pressure',
            'tsun': 'sunshine',
            'cldc': 'cloud_cover'
        })

        dfs.append(df_city)

    return pd.concat(dfs, ignore_index=True)

# Example
meteostat_all = load_meteostat_city_csvs('../data/processed')
meteostat_all.head()
meteostat_all.city.unique()

# Dem Dataframe das label picnic_weather hinzufügen, damit er mit weather_FS vergleichbar ist
# Dazu muss ich die Daten von weather_FS und meteostat_all anhand von city und DATE mergen. 
# Da die Daten von meteostat_all nur von 2000-2010 sind, muss ich den weather_FS DataFrame 
# auf diesen Zeitraum filtern, bevor ich merge.
weather_FS_filtered = weather_FS[(weather_FS['DATE'] >= '2000-01-01') & 
                                 (weather_FS['DATE'] <= '2010-01-01')]
meteostat_all = pd.merge(meteostat_all, 
                         weather_FS_filtered[['DATE', 'city', 'picnic_weather']], on=['DATE', 'city'], how='left')

# Füge noch eine spalte mit der Monatsnummer hinzu, damit ich die Daten von meteostat_all mit weather_FS vergleichen kann, da weather_FS auch eine Spalte MONTH hat
meteostat_all['MONTH'] = meteostat_all['DATE'].dt.month

# Save the combined DataFrame to a new CSV file
# meteostat_all.to_csv('../data/processed/weather_meteostat_lat_lon_elevation.csv', index=False)
# print("Combined Meteostat data saved to ../data/processed/weather_meteostat_lat_lon_elevation.csv")



In [ ]:
overview(meteostat_all)
overview(weather_FS)

In [ ]:
# lat, lon = city_coords['BASEL']
# weather_meteostat(lat, lon)

# weather_meteostat2('BASEL', save_csv=True, csv_path='../data/processed/weather_meteostat_basel_2000-2010.csv')
weather_meteostat2('STOCKHOLM', save_csv=True, csv_path='../data/processed/weather_meteostat_stockholm_2000-2010.csv')
weather_meteostat2('TOURS', save_csv=True, csv_path='../data/processed/weather_meteostat_tours_2000-2010.csv')

# City: SONNBLICK
# Elevation für SONNBLICK: 2427.0 m


In [ ]:

for cit in weather_FS.city.unique():
    print('City:', cit)
    weather_meteostat2(cit, save_csv=True, csv_path=f'../data/processed/weather_meteostat_{cit.lower()}_2000-2010.csv')

In [ ]:
def plot_compare_datasets(city, weather_df, csv_path, csv_date_col='DATE', csv_temp_col='temp_mean'):
    # Filter the weather_df for the specified city
    city_weather = weather_df[weather_df['city'] == city].copy()
    if city_weather.empty:
        raise ValueError(f"City '{city}' not found in weather_df")

    # Ensure the DATE column is in datetime format
    city_weather['DATE'] = pd.to_datetime(city_weather['DATE'])

    # Load the comparison dataset from CSV
    csv_df = pd.read_csv(csv_path, parse_dates=[csv_date_col])
    if csv_date_col not in csv_df.columns:
        raise ValueError(f"Date column '{csv_date_col}' not found in CSV file")
    if csv_temp_col not in csv_df.columns:
        raise ValueError(f"Temperature column '{csv_temp_col}' not found in CSV file")

    csv_df[csv_date_col] = pd.to_datetime(csv_df[csv_date_col])

    # Plotting
    plt.figure(figsize=(14, 6))

    # Plot temperature from weather_df
    plt.plot(city_weather['DATE'], city_weather[csv_temp_col], label='weather_FS '+csv_temp_col, color='blue')

    # Plot temperature from the CSV file
    plt.plot(csv_df[csv_date_col], csv_df[csv_temp_col], label=f'CSV {csv_temp_col} ({csv_path})', color='orange')

    plt.title(f'Temperature Comparison for {city}')
    plt.xlabel('Date')
    plt.ylabel('Temperature (°C)')
    plt.legend()
    plt.grid()
    plt.show()

In [ ]:
# plot_compare_datasets('BASEL', weather_FS, '../data/processed/weather_meteostat_basel_2000-2010.csv')
plot_compare_datasets('BASEL', weather_FS, '../data/processed/weather_meteostat_basel_2000-2010.csv',
                    #   csv_date_col='DATE', csv_temp_col='precipitation')
                    #   csv_date_col='DATE', csv_temp_col='temp_max')
                      csv_date_col='DATE', csv_temp_col='sunshine')
                    #   csv_date_col='DATE', csv_temp_col='cloud_cover')
                    #   csv_date_col='DATE', csv_temp_col='precipitation')


In [ ]:
df_basel

In [ ]:


# Specify location and time range
POINT = ms.Point(50.1155, 8.6842, 113)  # Try with your location
START = date(2001, 1, 1)
END = date(2001, 12, 31)

# Get nearby weather stations
stations = ms.stations.nearby(POINT, limit=4)

# Get daily data & perform interpolation
ts = ms.daily(stations, START, END)
df = ms.interpolate(ts, POINT).fetch()

# Plot line chart including average, minimum and maximum temperature
df.plot(y=[ms.Parameter.TEMP, ms.Parameter.TMIN, ms.Parameter.TMAX])
plt.show()

In [ ]:
df = weather_FS.copy()
# df = meteostat_all.copy()



---

## 4. Numerische Variablen analysieren



In [ ]:

# Statistiken für numerische Spalten
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if numeric_cols:
    print(f"📊 {len(numeric_cols)} numerische Spalten gefunden\n")
    display(df[numeric_cols].describe().round(2).T)
else:
    print("⚠️ Keine numerischen Spalten gefunden")



In [ ]:
plt.scatter(df['sunshine'], df['global_radiation'])
# Lineare Regression hinzufügen
sns.regplot(x='sunshine', y='global_radiation', data=df, scatter=False, color='red')
# return fit parameters a and b for the line global_radiation = a * sunshine + b
# a, b = np.polyfit(df['sunshine'], df['global_radiation'], 1)
a, b = (0.18, 0.4)
# a, b = (2.75/15, 0.4)
plt.title('Global Radiation vs. Sunshine')
print('Lin Fit: Global Radiation vs. Sunshine Funktion: global_radiation = a * sunshine + b')
print('a =', a)
print('b =', b)
plt.xlabel('Sunshine (hours)')
plt.ylabel('Global Radiation (W/m²)')
# plot the line global_radiation = a * sunshine + b
x = np.linspace(df['sunshine'].min(), df['sunshine'].max(), 100)
y = a * x + b 
plt.plot(x, y, color='red', label=f'Fit: global_radiation = {a:.2f} * sunshine + {b:.2f}')
plt.legend()
plt.grid()


In [ ]:
# Verteilungen der numerischen Variablen
if numeric_cols:
    n_cols = min(3, len(numeric_cols))
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
    axes = axes.flatten() if len(numeric_cols) > 1 else [axes]
    
    for idx, col in enumerate(numeric_cols):
        ax = axes[idx]
        df[col].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
        ax.set_title(f'{col}', fontsize=10, fontweight='bold')
        ax.grid(alpha=0.3)
    
    # Leere Subplots ausblenden
    for idx in range(len(numeric_cols), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()




---

## 5. Kategorische Variablen analysieren



In [ ]:

# Kategorische Spalten identifizieren
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

if categorical_cols:
    print(f"🏷️ {len(categorical_cols)} kategorische Spalten gefunden\n")
    
    cat_summary = pd.DataFrame({
        "Spalte": categorical_cols,
        "Unique Werte": [df[col].nunique() for col in categorical_cols],
        "Häufigste Werte": [df[col].value_counts().head(3).to_dict() for col in categorical_cols]
    })
    
    display(cat_summary)
else:
    print("⚠️ Keine kategorischen Spalten gefunden")


In [ ]:
# Top-Kategorien visualisieren (für Spalten mit wenigen Unique-Werten)
if categorical_cols:
    low_cardinality_cols = [col for col in categorical_cols if df[col].nunique() <= 20]
    
    if low_cardinality_cols:
        n_cols = min(2, len(low_cardinality_cols))
        n_rows = (len(low_cardinality_cols) + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
        axes = axes.flatten() if len(low_cardinality_cols) > 1 else [axes]
        
        for idx, col in enumerate(low_cardinality_cols):
            ax = axes[idx]
            value_counts = df[col].value_counts().head(10)
            value_counts.plot(kind='barh', ax=ax, color='steelblue')
            ax.set_title(f'Top Kategorien: {col}', fontsize=10, fontweight='bold')
            ax.set_xlabel('Anzahl')
            ax.grid(alpha=0.3, axis='x')
        
        # Leere Subplots ausblenden
        for idx in range(len(low_cardinality_cols), len(axes)):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()




---

## 6. Korrelationsanalyse



In [ ]:

if len(numeric_cols) > 1:
    # Korrelationsmatrix
    corr_matrix = df[numeric_cols].corr()
    
    # Heatmap
    fig, ax = plt.subplots(figsize=(10, 8), dpi=100)
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)  # Obere Dreiecksmatrix maskieren
    
    sns.heatmap(
        corr_matrix, 
        mask=mask,
        annot=True, 
        fmt='.2f',
        cmap='RdBu', 
        center=0, 
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8},
        ax=ax
    )
    
    ax.set_title('Korrelationsmatrix (numerische Variablen)', 
                 fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    
    # Stärkste Korrelationen finden
    print("\n🔗 Stärkste Korrelationen (|r| > 0.5):\n")
    corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.5:
                corr_pairs.append({
                    'Variable 1': corr_matrix.columns[i],
                    'Variable 2': corr_matrix.columns[j],
                    'Korrelation': corr_matrix.iloc[i, j]
                })
    
    if corr_pairs:
        corr_df = pd.DataFrame(corr_pairs).sort_values('Korrelation', 
                                                        key=abs, 
                                                        ascending=False)
        display(corr_df)
    else:
        print("Keine starken Korrelationen gefunden.")
else:
    print("⚠️ Nicht genügend numerische Variablen für Korrelationsanalyse")




---

## 7. Fehlende Werte visualisieren



In [ ]:

missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

if len(missing_data) > 0:
    fig, ax = plt.subplots(figsize=(10, max(6, len(missing_data) * 0.4)))
    
    missing_percent = (missing_data / len(df) * 100)
    missing_percent.plot(kind='barh', ax=ax, color='coral')
    
    ax.set_title('Fehlende Werte pro Spalte', fontsize=12, fontweight='bold')
    ax.set_xlabel('Prozent fehlend (%)')
    ax.grid(alpha=0.3, axis='x')
    
    # Werte als Text hinzufügen
    for i, v in enumerate(missing_percent):
        ax.text(v + 0.5, i, f'{v:.1f}%', va='center')
    
    plt.tight_layout()
    plt.show()
else:
    print("✅ Keine fehlenden Werte im Dataset!")




---

## 8. Zusammenfassung & Bewertung



In [ ]:

print("=" * 80)
print("📝 DATASET-BEWERTUNG FÜR PORTFOLIO-PROJEKT")
print("=" * 80)

# Qualitätskriterien berechnen
data_quality = {
    "Datengröße": "✅ Gut" if len(df) > 5_000 else "⚠️ Klein" if len(df) > 1_000 else "❌ Zu klein",
    "Vollständigkeit": f"✅ {(1 - df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100:.1f}%",
    "Duplikate": "✅ Keine" if df.duplicated().sum() == 0 else f"⚠️ {df.duplicated().sum()} gefunden",
    "Variablenvielfalt": f"{'✅' if df.shape[1] >= 10 else '⚠️'} {df.shape[1]} Spalten",
    "Numerische Features": f"{'✅' if len(numeric_cols) >= 5 else '⚠️'} {len(numeric_cols)} Spalten",
    "Kategorische Features": f"{'✅' if len(categorical_cols) >= 1 else '⚠️'} {len(categorical_cols)} Spalten"
}

for criterion, status in data_quality.items():
    print(f"{criterion:.<30} {status}")

print("\n" + "=" * 80)
print("💡 EMPFEHLUNG:")
print("=" * 80)

score = sum([
    len(df) > 5000,
    df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) < 0.1,
    df.duplicated().sum() < len(df) * 0.05,
    df.shape[1] >= 10,
    len(numeric_cols) >= 5
])

if score >= 4:
    print("✅ SEHR GUT - Dieses Dataset eignet sich hervorragend für ein Portfolio-Projekt!")
elif score >= 3:
    print("👍 OK - Dieses Dataset ist für ein Portfolio-Projekt geeignet.")
else:
    print("⚠️ BEDINGT - Überlege, ob dieses Dataset für dein Projekt ausreicht.")

print("\n📌 Nächste Schritte:")
print("   1. Definiere deine Forschungsfrage")
print("   2. Identifiziere relevante Features")
print("   3. Plane deine Analysestrategie")
print("   4. Beginne mit Feature Engineering & Modeling")
